In [1]:
from retrievers.vector_retriever import VectorRetriever
from retrievers.fulltext_retriever import FullTextRetriever
from retrievers.hybrid_retriever import HybridRetriever
from retrievers.text2cyper import Text2CypherRetriever
from neo4j_manager import Neo4jManager
# from agents.agentic_rag import AgenticRAG
from agents.multi_agent_system import MultiAgentSystem
from agents.question_descomposer import QuestionDecomposer


In [2]:
neo4j = Neo4jManager()


In [ ]:
query = "Who is Momotaro?"

extra_match = """
OPTIONAL MATCH (node)-[:HAS_ROLE]->(r:Role)
"""

return_projection={
	"name": "node.name",
	"description": "node.description",
	"role": "r.name"
}
retriever = VectorRetriever(neo4j, "agent_embeddings", return_projection, extra_match)
results = retriever.retrieve(query)

print(f"Vector search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]} [{r["role"]}]: {r["description"]}\n")


In [ ]:
query = "millet dumpling"
return_fields = {
    "description": "node.description",
    "name": "node.name"
}
retriever = FullTextRetriever(neo4j, "event_fulltext", "Event", "description", return_fields)
results = retriever.retrieve(query)

print(f"Fulltext search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]}: {r["description"]}\n")


In [ ]:
query = "How did Momotaro convince the animals to join him on his journey?"
return_fields = {
    "description": "node.description",
    "name": "node.name"
}
retriever = HybridRetriever(neo4j, "agent_embeddings", "agent_fulltext", "Agent", "description", vector_weight=0.5, return_projection=return_fields, include_score=True)
results = retriever.retrieve(query, 10)

print(f"Hybrid search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]}: {r["description"]}\n")


In [ ]:
# What happens in Momotaro, in order, and which characters are involved in each event?
cypher_query = """
MATCH (f:Folktale {url: $url})-[:HAS_EVENT]->(e:Event)
OPTIONAL MATCH (e)-[:HAS_AGENT]->(a:Agent)
RETURN e.order AS order,
       e.name AS event,
       collect(a.name) AS agents
ORDER BY order
"""

params = {
    "url": "https://fairytalez.com/momotaro/"
}

results = neo4j.execute_query(cypher_query, params)

for idx, row in enumerate(results):
    print(f"- {idx}. {row["event"]}: {row["agents"]}")


In [ ]:
# Which characters are most important in the story based on how many events they appear in?
cypher_query = """
MATCH (a:Agent)
OPTIONAL MATCH (a)<-[:HAS_AGENT]-(e:Event)
RETURN a.name AS agent,
       count(DISTINCT e) AS event_count
ORDER BY event_count DESC
LIMIT 10
"""

results = neo4j.execute_query(cypher_query)

for idx, row in enumerate(results):
    print(f"- {row["agent"]}: {row["event_count"]}")



In [ ]:
# What happens after the specific event where Momotaro gives the dumpling to the dog?
cypher_query = """
MATCH (e:Event {name: $name})-[:POST_EVENT]->(next:Event)
RETURN e.name AS current_event,
       next.name AS next_event
"""

params = {
    "name": "Momotaro gives dog a dumpling."
}

results = neo4j.execute_query(cypher_query, params)

for row in results:
    print(f"- {row['current_event']} -> {row['next_event']}")


In [ ]:
# How many folktales come from Japan?
cypher_query = """
MATCH (f:Folktale)-[:FROM_NATION]->(n:Nation {name: $name})
RETURN count(f) AS total_folktales
"""

params = {
    "name": "japanese"
}

results = neo4j.execute_query(cypher_query, params)

for row in results:
    print(row["total_folktales"])


In [ ]:
# Who are Momotaro's friends?
cypher_query = """
MATCH (m:Agent {name: $name})-[r:RELATIONSHIP]->(a:Agent)
WHERE r.type = "friend"
RETURN a.name AS friend
"""

params = {
    "name": "Momotaro"
}

results = neo4j.execute_query(cypher_query, params)

for row in results:
    print(row["friend"])


In [ ]:
cypher_query = """
MATCH (f:Folktale)
RETURN f.title as title,
       f.url as url
"""

results = neo4j.execute_query(cypher_query)

for row in results:
    print(row)
    

In [ ]:
text2cypher = Text2CypherRetriever(neo4j)

text2cypher.add_few_shot_example(
    "What happens in Momotaro, in order, and which characters are involved in each event?",
    """
    MATCH (f:Folktale {url: "https://fairytalez.com/momotaro/"})-[:HAS_EVENT]->(e:Event)
    OPTIONAL MATCH (e)-[:HAS_AGENT]->(a:Agent)
    RETURN e.order AS order,
           e.name AS event,
           collect(a.name) AS agents
    ORDER BY order
    """
)

text2cypher.add_few_shot_example(
    "Which characters are most important in the story based on how many events they appear in?",
    """
    MATCH (a:Agent)
    OPTIONAL MATCH (a)<-[:HAS_AGENT]-(e:Event)
    RETURN a.name AS agent,
           count(DISTINCT e) AS event_count
    ORDER BY event_count DESC
    LIMIT 10
    """
)

text2cypher.add_few_shot_example(
    "How many folktales come from Japan?",
    """
    MATCH (f:Folktale)-[:FROM_NATION]->(n:Nation {name: "Japan"})
    RETURN count(f) AS total_folktales
    """
)

text2cypher.add_few_shot_example(
    "Who are Momotaro's friends?",
    """
    MATCH (m:Agent {name: "Momotaro"})-[r:RELATIONSHIP]->(a:Agent)
    WHERE r.type = "friend"
    RETURN a.name AS friend
    """
)

text2cypher.add_few_shot_example(
    "What happens after the specific event where Momotaro gives the dumpling to the dog?",
    """
    MATCH (e:Event {name: "Momotaro gives dog a dumpling."})-[:POST_EVENT]->(next:Event)
    RETURN e.name AS current_event,
           next.name AS next_event
    """
)


In [ ]:
# Consulta de conteo sobre nodo de dominio real
cypher, results = text2cypher.retrieve("How many Event nodes are in the graph?")

print("Cypher generado:")
print(f"  {cypher}\n")
print("Resultados:")
for r in results:
    print(f"  {r}")



In [ ]:
# Traversal: qué instituciones aparecen en el grafo
cypher, results = text2cypher.retrieve("How many characters have the role of helper?")

print("Cypher generado:")
print(f"  {cypher}\n")
print("Resultados:")
for r in results:
    print(f"  {r}")
    

In [5]:
rag = MultiAgentSystem(neo4j)


In [ ]:
# How did Momotaro convince the animals to join him on his journey?
# How many Event nodes are in the graph?"
# How many characters have the role of helper?

result = rag.answer("How did Momotaro convince the animals to join him on his journey?", "test1")

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)


In [ ]:
# How did Momotaro convince the animals to join him on his journey?
# How many Event nodes are in the graph?"
# How many characters have the role of helper?

result = rag.answer("How many characters have the role of helper?", "test4")

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)



In [6]:
# In stories where there is a hero, what is the main test he must overcome?

result = rag.answer("Where do the ogres live in the story of Momotaro?", "test")

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)



sub_questions=[] should_decompose=False
['Where do the ogres live in the story of Momotaro?']

========== ITERATION 1 ==========

  ========== QUESTION 1 ==========
  Question: Where do the ogres live in the story of Momotaro?



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "place_tool",
    "args": {
      "query": "Where do the ogres live in the story of Momotaro?"
    },
    "id": "c0a26833-6f4e-4939-9d4f-9af6c97cae28",
    "type": "tool_call"
  }
]
sub_questions=[] should_decompose=False
['Where do the ogres live in the story of Momotaro?']

========== ITERATION 1 ==========

  ========== QUESTION 1 ==========
  Question: Where do the ogres live in the story of Momotaro?



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "hybrid_search",
    "args": {
      "query": "Where do the ogres live in the story of Momotaro?"
    },
    "id": "e938b3ae-9894-4d8e-a65d-5b89dde1aeab",
    "type": "tool_call"
  }
]


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=4, column=13, offset=60>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 60, 'line': 4, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL () {\n            // Vector search\n            CALL db.index.vector.queryNodes($vector_index, $top_k, $query_embedding)\n            YIELD node, score\n            \n            WITH collect({node: node, score: score}) AS nodes, max(score) AS maxScore, min(score) AS minScore\n            UNWIND nodes AS n\n            RET

hybrid_search
{'vector_search': StructuredTool(name='vector_search', description='Use it for semantic search over folktale knowledge when the user asks open-ended, conceptual or descriptive questions. Ideal for themes, meanings, summaries, story interpretation, narrative elements or sysbolism.This tool performs pure vector similarity search and returns the most relevant text passages ranked by semantic relevance.', args_schema=<class 'agents.rag_tool_factory.QueryInput'>, func=<bound method RAGToolFactory._vector_search of <agents.rag_tool_factory.RAGToolFactory object at 0x0000023ABB8F9A60>>), 'hybrid_search': StructuredTool(name='hybrid_search', description='Use when the query contains BOTH semantic intent and explicit signals such as named folktales, characters, places, cultures objects, or partial keywords. Best for mixed queries that require combining keyword matching with semantic retrieval, such as specific story searches with context or partial information. Returns relevant pas

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=4, column=13, offset=60>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 60, 'line': 4, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL () {\n            // Vector search\n            CALL db.index.vector.queryNodes($vector_index, $top_k, $query_embedding)\n            YIELD node, score\n            \n            WITH collect({node: node, score: score}) AS nodes, max(score) AS maxScore, min(score) AS minScore\n            UNWIND nodes AS n\n            RET

hybrid_search
{'vector_search': StructuredTool(name='vector_search', description='Use it for semantic search over folktale knowledge when the user asks open-ended, conceptual or descriptive questions. Ideal for themes, meanings, summaries, story interpretation, narrative elements or sysbolism.This tool performs pure vector similarity search and returns the most relevant text passages ranked by semantic relevance.', args_schema=<class 'agents.rag_tool_factory.QueryInput'>, func=<bound method RAGToolFactory._vector_search of <agents.rag_tool_factory.RAGToolFactory object at 0x0000023ABB8F9A60>>), 'hybrid_search': StructuredTool(name='hybrid_search', description='Use when the query contains BOTH semantic intent and explicit signals such as named folktales, characters, places, cultures objects, or partial keywords. Best for mixed queries that require combining keyword matching with semantic retrieval, such as specific story searches with context or partial information. Returns relevant pas

...

hybrid_search
{'vector_search': StructuredTool(name='vector_search', description='Use it for semantic search over folktale knowledge when the user asks open-ended, conceptual or descriptive questions. Ideal for themes, meanings, summaries, story interpretation, narrative elements or sysbolism.This tool performs pure vector similarity search and returns the most relevant text passages ranked by semantic relevance.', args_schema=<class 'agents.rag_tool_factory.QueryInput'>, func=<bound method RAGToolFactory._vector_search of <agents.rag_tool_factory.RAGToolFactory object at 0x0000023ABB8F9A60>>), 'hybrid_search': StructuredTool(name='hybrid_search', description='Use when the query contains BOTH semantic intent and explicit signals such as named folktales, characters, places, cultures objects, or partial keywords. Best for mixed queries that require combining keyword matching with semantic retrieval, such as specific story searches with context or partial information. Returns relevant pas

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL db.index.vector.queryNodes($index_name, $top_k, $query_embedding)\n        YIELD node, score\n\n        \n\n        RETURN node.name AS name, node.description AS description, node.type AS type\n        ORDER BY score DESC\n        '


vector_search
{'vector_search': StructuredTool(name='vector_search', description='Use it for semantic search over folktale knowledge when the user asks open-ended, conceptual or descriptive questions. Ideal for themes, meanings, summaries, story interpretation, narrative elements or sysbolism.This tool performs pure vector similarity search and returns the most relevant text passages ranked by semantic relevance.', args_schema=<class 'agents.rag_tool_factory.QueryInput'>, func=<bound method RAGToolFactory._vector_search of <agents.rag_tool_factory.RAGToolFactory object at 0x0000023ABB8F9A60>>), 'hybrid_search': StructuredTool(name='hybrid_search', description='Use when the query contains BOTH semantic intent and explicit signals such as named folktales, characters, places, cultures objects, or partial keywords. Best for mixed queries that require combining keyword matching with semantic retrieval, such as specific story searches with context or partial information. Returns relevant pas

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL db.index.vector.queryNodes($index_name, $top_k, $query_embedding)\n        YIELD node, score\n\n        \n\n        RETURN node.name AS name, node.description AS description, node.type AS type\n        ORDER BY score DESC\n        '


vector_search
{'vector_search': StructuredTool(name='vector_search', description='Use it for semantic search over folktale knowledge when the user asks open-ended, conceptual or descriptive questions. Ideal for themes, meanings, summaries, story interpretation, narrative elements or sysbolism.This tool performs pure vector similarity search and returns the most relevant text passages ranked by semantic relevance.', args_schema=<class 'agents.rag_tool_factory.QueryInput'>, func=<bound method RAGToolFactory._vector_search of <agents.rag_tool_factory.RAGToolFactory object at 0x0000023ABB8F9A60>>), 'hybrid_search': StructuredTool(name='hybrid_search', description='Use when the query contains BOTH semantic intent and explicit signals such as named folktales, characters, places, cultures objects, or partial keywords. Best for mixed queries that require combining keyword matching with semantic retrieval, such as specific story searches with context or partial information. Returns relevant pas

In [ ]:
result = rag.answer("Hi, my name is Matt", "holamatt")

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)


In [ ]:
result = rag.answer("What is the weather", "duh")

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)


In [ ]:
neo4j.close()
